In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('../data/names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
#build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print (vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# building the dataset
block_size = 3 # context length: how many characters do we take to predict the next one

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # crop and append
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

X_tr, Y_tr = build_dataset(words[:n1])      # 80%
X_dev, Y_dev = build_dataset(words[n1:n2])  # 10%
X_te, Y_te = build_dataset(words[n2:])      # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# utility function for comparing manual gradients to PyTorch gradiends
def compare_gradients(s, d_t, t):
    exact = torch.all(d_t == t.grad).item()
    approx = torch.allclose(d_t, t.grad)
    max_difference = (d_t - t.grad).abs().max().item()

    print(f'{s:15s} | exact: {str(exact):5s} | approximate: {str(approx):5s} | max_difference: {max_difference}')

In [6]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # number of neurons in the hidden layer of MLP

g = torch.Generator().manual_seed(2147483647) # used for reproducibility
C = torch.randn((vocab_size, n_embd),             generator=g)

# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3) / ((n_embd * block_size) ** 0.5) # kaiming normalization (std)
b1 = torch.randn((n_hidden),                      generator=g) * 0.1 # not needed if we use batch normalization bias (bn_bias), using it now only for fun

# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn((vocab_size),                    generator=g) * 0.1

# BatchNorm parameters
bn_gain = torch.ones((1, n_hidden)) * 0.1 + 1.0
bn_bias = torch.zeros((1, n_hidden)) * 0.1

# Note: Initializing many of these parameter with e.g. all zeros could mas and incorrect implementation of the backward pass, so we are initializing them in non-standard way

parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

4137


In [7]:
batch_size = 32

# minibatch construct
ix = torch.randint(0, X_tr.shape[0], (batch_size,), generator=g)
X_batch, Y_batch = X_tr[ix], Y_tr[ix] # batch X, Y

In [8]:
# forward pass, "chunked" into smaller steps that are possible to backward one at a time

emb = C[X_batch] # embed the characters into vectors
emb_cat = emb.view(emb.shape[0], -1) # concatenate the vectors

# linear layer 1
h_preact_bn = emb_cat @ W1 # hidden layer pre-activation

# BatchNorm layer
bn_mean_i = 1/h_preact_bn.sum(0, keepdim=True)
bn_diff = h_preact_bn - bn_mean_i # difference
bn_diff_sq = bn_diff ** 2 # difference squared
bn_var = 1/(batch_size - 1) * (bn_diff_sq).sum(0, keepdim=True) # Bessel's correction (dividing by batch_size - 1, not batch_size)
bn_var_inv = (bn_var + 1e-5) ** -0.5
bn_raw = bn_diff * bn_var_inv
h_preact = bn_gain * bn_raw + bn_bias

# non-linearity
h = torch.tanh(h_preact) # hidden layer

# linear layer 2
logits = h @ W2 + b2 # output layer

# cross entropy loss (same as F.cross_entropy(logits, Y_batch))
logits_max = logits.max(1, keepdim=True).values
norm_logits = logits - logits_max # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1 # can't get backprop to be exact if using (1.0 / counts_sum)
probs = counts * counts_sum_inv
log_probs = probs.log()
loss = -log_probs[range(batch_size), Y_batch].mean()

# PyTorch backward pass
for p in parameters:
    p.grad = None

for t in [log_probs, probs, counts, counts_sum, counts_sum_inv, norm_logits, logits_max, logits, h, h_preact, bn_raw, bn_var_inv, bn_var, bn_diff_sq, bn_diff, h_preact_bn, bn_mean_i, emb_cat, emb]:
    t.retain_grad()

loss.backward()
loss

tensor(3.4764, grad_fn=<NegBackward0>)